In [1]:
import pandas as pd

file_path = "../data/raw/EMS_Incident_Dispatch_Data.csv"
df = pd.read_csv(file_path, nrows=1000000)

df.head()

,CAD_INCIDENT_ID,INCIDENT_DATETIME,INITIAL_CALL_TYPE,INITIAL_SEVERITY_LEVEL_CODE,FINAL_CALL_TYPE,FINAL_SEVERITY_LEVEL_CODE,FIRST_ASSIGNMENT_DATETIME,VALID_DISPATCH_RSPNS_TIME_INDC,DISPATCH_RESPONSE_SECONDS_QY,FIRST_ACTIVATION_DATETIME,...,ZIPCODE,POLICEPRECINCT,CITYCOUNCILDISTRICT,COMMUNITYDISTRICT,COMMUNITYSCHOOLDISTRICT,CONGRESSIONALDISTRICT,REOPEN_INDICATOR,SPECIAL_EVENT_INDICATOR,STANDBY_INDICATOR,TRANSFER_INDICATOR
0,50010612,01/01/2005 02:02:20 AM,INJURY,5,INJURY,5,01/01/2005 02:02:37 AM,Y,17,01/01/2005 02:02:54 AM,...,NaN,NaN,NaN,NaN,NaN,NaN,N,N,N,N
1,50010718,01/01/2005 02:20:42 AM,DRUG,4,DRUG,4,01/01/2005 02:27:42 AM,Y,420,01/01/2005 02:29:03 AM,...,NaN,NaN,NaN,NaN,NaN,NaN,N,N,N,N
2,50010853,01/01/2005 02:45:39 AM,DRUG,4,DRUG,4,01/01/2005 02:53:06 AM,Y,447,01/01/2005 03:02:29 AM,...,NaN,NaN,NaN,NaN,NaN,NaN,N,N,N,N
3,50011297,01/01/2005 04:02:32 AM,INJURY,5,INJURY,5,NaN,N,0,NaN,...,10467.0,47.0,12.0,212.0,11.0,16.0,N,N,N,N
4,50011923,01/01/2005 06:50:35 AM,MVAINJ,4,MVAINJ,4,01/01/2005 06:55:46 AM,Y,311,01/01/2005 06:55:54 AM,...,NaN,NaN,NaN,NaN,NaN,NaN,N,N,N,N


In [2]:
df.dtypes

CAD_INCIDENT_ID                     int64
INCIDENT_DATETIME                  object
INITIAL_CALL_TYPE                  object
INITIAL_SEVERITY_LEVEL_CODE         int64
FINAL_CALL_TYPE                    object
FINAL_SEVERITY_LEVEL_CODE           int64
FIRST_ASSIGNMENT_DATETIME          object
VALID_DISPATCH_RSPNS_TIME_INDC     object
DISPATCH_RESPONSE_SECONDS_QY       object
FIRST_ACTIVATION_DATETIME          object
FIRST_ON_SCENE_DATETIME            object
VALID_INCIDENT_RSPNS_TIME_INDC     object
INCIDENT_RESPONSE_SECONDS_QY       object
INCIDENT_TRAVEL_TM_SECONDS_QY      object
FIRST_TO_HOSP_DATETIME             object
FIRST_HOSP_ARRIVAL_DATETIME        object
INCIDENT_CLOSE_DATETIME            object
HELD_INDICATOR                     object
INCIDENT_DISPOSITION_CODE          object
BOROUGH                            object
INCIDENT_DISPATCH_AREA             object
ZIPCODE                           float64
POLICEPRECINCT                    float64
CITYCOUNCILDISTRICT               

## Sample preprocessing

In [3]:
# clean data types

date_cols = [
    "INCIDENT_DATETIME",
    "FIRST_ASSIGNMENT_DATETIME",
    "FIRST_ACTIVATION_DATETIME",
    "FIRST_ON_SCENE_DATETIME",
    "INCIDENT_CLOSE_DATETIME",
    "FIRST_TO_HOSP_DATETIME",
    "FIRST_HOSP_ARRIVAL_DATETIME"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

numeric_cols = [
    "INCIDENT_RESPONSE_SECONDS_QY",
    "DISPATCH_RESPONSE_SECONDS_QY",
    "INCIDENT_TRAVEL_TM_SECONDS_QY"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols].dtypes

df.dtypes

C:\Users\Laura Zillmer\AppData\Local\Temp\ipykernel_29856\534144769.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")
C:\Users\Laura Zillmer\AppData\Local\Temp\ipykernel_29856\534144769.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


CAD_INCIDENT_ID                            int64
INCIDENT_DATETIME                 datetime64[ns]
INITIAL_CALL_TYPE                         object
INITIAL_SEVERITY_LEVEL_CODE                int64
FINAL_CALL_TYPE                           object
FINAL_SEVERITY_LEVEL_CODE                  int64
FIRST_ASSIGNMENT_DATETIME         datetime64[ns]
VALID_DISPATCH_RSPNS_TIME_INDC            object
DISPATCH_RESPONSE_SECONDS_QY             float64
FIRST_ACTIVATION_DATETIME         datetime64[ns]
FIRST_ON_SCENE_DATETIME           datetime64[ns]
VALID_INCIDENT_RSPNS_TIME_INDC            object
INCIDENT_RESPONSE_SECONDS_QY             float64
INCIDENT_TRAVEL_TM_SECONDS_QY            float64
FIRST_TO_HOSP_DATETIME            datetime64[ns]
FIRST_HOSP_ARRIVAL_DATETIME       datetime64[ns]
INCIDENT_CLOSE_DATETIME           datetime64[ns]
HELD_INDICATOR                            object
INCIDENT_DISPOSITION_CODE                 object
BOROUGH                                   object
INCIDENT_DISPATCH_AR

In [4]:
# clean categorical variables
df["BOROUGH"] = df["BOROUGH"].str.strip().str.upper()
df["INITIAL_CALL_TYPE"] = df["INITIAL_CALL_TYPE"].str.strip().str.upper()

# filter to valid response times
df = df[df["VALID_INCIDENT_RSPNS_TIME_INDC"] == "Y"]

# drop missing response times and incident datetimes
df = df.dropna(subset=["INCIDENT_RESPONSE_SECONDS_QY", "INCIDENT_DATETIME"])

# remove invalid response times
df = df[
    (df["INCIDENT_RESPONSE_SECONDS_QY"] > 1) &
    (df["INCIDENT_RESPONSE_SECONDS_QY"] < 3600) 
]

# create time features
df["hour"] = df["INCIDENT_DATETIME"].dt.hour
df["date"] = df["INCIDENT_DATETIME"].dt.date

# create demand variable
call_volume = df.groupby(["date", "hour"]).size().reset_index(name="call_volume")
df = df.merge(call_volume, on=["date", "hour"], how="left")

## Full dataset preprocessing

In [5]:
# # process full dataset in chunks
# file_path = "../data/raw/EMS_Incident_Dispatch_Data.csv"
# output_path = "../data/processed/ems_cleaned.csv"

# date_cols = [
#     "INCIDENT_DATETIME",
#     "FIRST_ASSIGNMENT_DATETIME",
#     "FIRST_ACTIVATION_DATETIME",
#     "FIRST_ON_SCENE_DATETIME",
#     "INCIDENT_CLOSE_DATETIME",
#     "FIRST_TO_HOSP_DATETIME",
#     "FIRST_HOSP_ARRIVAL_DATETIME"
# ]

# numeric_cols = [
#     "INCIDENT_RESPONSE_SECONDS_QY",
#     "DISPATCH_RESPONSE_SECONDS_QY",
#     "INCIDENT_TRAVEL_TM_SECONDS_QY"
# ]

# first_chunk = True

# for chunk in pd.read_csv(file_path, chunksize=100000):
#     # clean data types
#     for col in date_cols:
#         chunk[col] = pd.to_datetime(chunk[col], errors="coerce")

#     for col in numeric_cols:
#         chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

#     # clean categorical variables
#     chunk["BOROUGH"] = chunk["BOROUGH"].str.strip().str.upper()
#     chunk["INITIAL_CALL_TYPE"] = chunk["INITIAL_CALL_TYPE"].str.strip().str.upper()

#     # filter to valid response times
#     chunk = chunk[chunk["VALID_INCIDENT_RSPNS_TIME_INDC"] == "Y"]

#     # drop missing response times
#     chunk = chunk.dropna(subset=["INCIDENT_RESPONSE_SECONDS_QY", "INCIDENT_DATETIME"])

#     # save cleaned chunk
#     if first_chunk:
#         chunk.to_csv(output_path, index=False, mode="w")
#         first_chunk = False
#     else:
#         chunk.to_csv(output_path, index=False, mode="a", header=False)

## Feature engineering on full dataset

In [6]:
# df = pd.read_csv("../data/processed/ems_cleaned.csv")

# df["INCIDENT_DATETIME"] = pd.to_datetime(df["INCIDENT_DATETIME"], errors="coerce")

# # create time features
# df["hour"] = df["INCIDENT_DATETIME"].dt.hour
# df["date"] = df["INCIDENT_DATETIME"].dt.date

# # create demand variable
# call_volume = df.groupby(["date", "hour"]).size().reset_index(name="call_volume")
# df = df.merge(call_volume, on=["date", "hour"], how="left")

## Save cleaned dataset

In [7]:
# save cleaned dataset
df.to_csv("../data/processed/ems_cleaned.csv", index=False)